# Sales Objective: Completed-Cycle Learning Notebook

**Business job:** Generate delivered customer outcomes with acceptable revenue return.

This notebook follows the objective from raw Meta and WhatsApp evidence through
objective-specific KPIs, Empirical Bayes with an explicit benchmark hierarchy,
efficiency, deterministic action, budget allocation, child-entity evidence, and
conversation diagnostics.

It is an explanatory report for a completed cycle. It does not optimize a live
campaign and does not ask an LLM to calculate scores or budget.

## 1. Scope And Reading Rules

1. `objective` is the only campaign grouping used for evaluation.
2. Paid-attributed WhatsApp outcomes are treated as complete for this MVP.
3. Organic/direct conversations are outside paid campaign scoring.
4. Active and stuck-pending outcomes remain visible but are excluded from mature rates.
5. The primary score measures outcome effectiveness; efficiency remains separate.
6. Small samples lean toward selected peers and always show a range. Same-objective
   peers are preferred; Awareness and Engagement may share Link CTR evidence when
   their own objective has too few peers.
7. Conversation signals create a separate quality score and can prioritize exploration.
   Primary outcome scores and final actions remain driven by structured evidence.
8. Budget is assigned only to campaigns; child levels guide execution without
   double-counting the same money.

In [1]:
from pathlib import Path
import sys

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(path for path in candidates if (path / "src_mvp").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src_mvp.config import load_budget_policy, load_objectives
from src_mvp.pipeline import run_mvp
from src_mvp.statistics import fit_beta_prior

FOCUS_OBJECTIVE = "OUTCOME_SALES"
result = run_mvp(output_directory=ROOT / "src_mvp" / "outputs")
registry = load_objectives()
policy = load_budget_policy()
contract = registry.objectives[FOCUS_OBJECTIVE]
all_campaigns = result.scorecards["campaign"].copy()
campaigns = all_campaigns.loc[
    all_campaigns["objective"].eq(FOCUS_OBJECTIVE)
].copy()
allocations = {item.campaign_id: item for item in result.allocations}
objective_tests = [
    item for item in result.exploration_tests
    if item.objective == FOCUS_OBJECTIVE
]

DECISION_COLORS = {
    "scale": "#16815d",
    "hold": "#d68c16",
    "kill": "#c44536",
    "insufficient_evidence": "#6b7280",
}
alt.data_transformers.disable_max_rows()

print(f"Repository: {ROOT}")
print(f"Objective: {FOCUS_OBJECTIVE}")
print(f"Campaigns in scope: {len(campaigns)}")

Repository: /Users/abdelmoo/Desktop/CAPI Analysis/Team2-MarketingExpert
Objective: OUTCOME_SALES
Campaigns in scope: 6


## 2. Objective Contract

The objective determines the business question and the one primary statistical score.
It also selects one efficiency metric, which remains a separate gate.

**Important limitation:** The primary score is customer-level: each mature customer is counted once within an entity, even when that customer has multiple conversations. Net ROAS is kept separate to represent revenue efficiency.

In [2]:
contract_table = pd.DataFrame([
    ["Business job", contract.business_job],
    ["Success question", contract.success_question],
    ["Primary metric", contract.primary_metric],
    ["Numerator", contract.primary_numerator],
    ["Denominator", contract.primary_denominator],
    ["Better primary direction", contract.primary_direction],
    ["Efficiency metric", contract.efficiency_metric],
    ["Better efficiency direction", contract.efficiency_direction],
    ["Minimum primary trials", contract.minimum_trials],
    ["Minimum peer entities", contract.minimum_peer_entities],
    ["Configured fallback group", contract.fallback_benchmark_group or "None"],
    ["Semantic metric", contract.semantic.metric],
    ["Semantic definition", contract.semantic.definition],
], columns=["Contract element", "Value"])
display(contract_table.style.hide(axis="index"))

Contract element,Value
Business job,Generate delivered customer outcomes with acceptable revenue return.
Success question,Did mature unique customers reach delivery better than comparable sales campaigns?
Primary metric,customer_delivered_rate
Numerator,delivered_customers
Denominator,mature_unique_customers
Better primary direction,higher
Efficiency metric,net_roas
Better efficiency direction,higher
Minimum primary trials,10
Minimum peer entities,2


## 3. Campaign Inventory And Delivery

Spend and delivery volume describe campaign size; they do not prove success. The
primary outcome score below evaluates quality relative to the selected compatible
benchmark. Portfolio-wide context remains descriptive only.

In [3]:
inventory = campaigns[[
    "campaign_name", "entity_status", "media_start", "media_end",
    "running_days", "active_days", "spend", "impressions", "link_clicks",
    "observed_conversations", "mature_conversations",
    "unresolved_conversations",
]].rename(columns={
    "campaign_name": "Campaign", "entity_status": "Status",
    "media_start": "First delivery", "media_end": "Last delivery",
    "running_days": "Calendar days", "active_days": "Days with delivery",
    "spend": "Spend", "impressions": "Impressions",
    "link_clicks": "Link clicks",
    "observed_conversations": "Paid conversations",
    "mature_conversations": "Mature",
    "unresolved_conversations": "Unresolved",
})
display(inventory.style.format({"Spend": "{:,.2f}", "Impressions": "{:,.0f}"}))

,Campaign,Status,First delivery,Last delivery,Calendar days,Days with delivery,Spend,Impressions,Link clicks,Paid conversations,Mature,Unresolved
0,Always-On Premium Acquisition,ACTIVE,2025-12-30 00:00:00,2026-06-27 00:00:00,180,180,"140,783.91","26,091,014",196800,111,104,7
4,Ramadan Suhoor Specials,COMPLETED,2026-02-28 00:00:00,2026-03-29 00:00:00,30,30,"32,293.38","4,601,950",81525,64,62,2
5,Ramadan Iftar Premium Bundles,COMPLETED,2026-02-28 00:00:00,2026-03-29 00:00:00,30,30,"57,095.26","8,339,290",122984,84,76,8
6,Eid Gifting Premium,COMPLETED,2026-03-22 00:00:00,2026-04-05 00:00:00,15,15,"14,976.24","2,967,717",43344,47,46,1
8,Summer Premium Launch,COMPLETED,2026-04-29 00:00:00,2026-06-27 00:00:00,60,60,"68,595.84","14,187,459",162229,97,90,7
10,Mid-Year Sale,COMPLETED,2026-05-15 00:00:00,2026-06-05 00:00:00,22,22,"17,552.49","3,378,126",49270,34,34,0


## 4. Raw Primary Evidence

The raw score is the directly observed rate:

```text
raw rate = objective successes / eligible objective trials
```

These counts remain beside every corrected score so the modeled result can always be
audited back to real evidence.

In [4]:
raw_evidence = campaigns[[
    "campaign_name", "score_successes", "score_trials", "raw_rate",
    "observed_conversations", "mature_conversations",
    "unresolved_conversations", "evidence_status",
]].rename(columns={
    "campaign_name": "Campaign", "score_successes": "Successes",
    "score_trials": "Eligible trials", "raw_rate": "Raw rate",
    "observed_conversations": "Observed conversations",
    "mature_conversations": "Mature conversations",
    "unresolved_conversations": "Unresolved conversations",
    "evidence_status": "Evidence status",
})
display(raw_evidence.style.format({"Raw rate": "{:.2%}"}, na_rep="Not available"))

maturity_long = inventory[["Campaign", "Mature", "Unresolved"]].melt(
    "Campaign", var_name="Outcome maturity", value_name="Conversations"
)
maturity_chart = alt.Chart(maturity_long).mark_bar().encode(
    y=alt.Y("Campaign:N", sort="-x", title=None),
    x=alt.X("Conversations:Q", title="WhatsApp conversations"),
    color=alt.Color(
        "Outcome maturity:N",
        scale=alt.Scale(domain=["Mature", "Unresolved"], range=["#218380", "#d9a441"]),
        legend=alt.Legend(orient="top"),
    ),
    tooltip=["Campaign", "Outcome maturity", "Conversations"],
).properties(width=760, height=max(120, len(campaigns) * 42))
display(maturity_chart)

,Campaign,Successes,Eligible trials,Raw rate,Observed conversations,Mature conversations,Unresolved conversations,Evidence status
0,Always-On Premium Acquisition,52.000000,103.000000,50.49%,111,104,7,sufficient
4,Ramadan Suhoor Specials,41.000000,62.000000,66.13%,64,62,2,sufficient
5,Ramadan Iftar Premium Bundles,42.000000,76.000000,55.26%,84,76,8,sufficient
6,Eid Gifting Premium,28.000000,46.000000,60.87%,47,46,1,sufficient
8,Summer Premium Launch,64.000000,90.000000,71.11%,97,90,7,sufficient
10,Mid-Year Sale,20.000000,34.000000,58.82%,34,34,0,sufficient


alt.Chart(...)

## 5. Leave-One-Out Peer Evidence And Prior

The benchmark hierarchy is explicit:

1. Use other campaigns with the same objective when at least two are valid.
2. If those peers are insufficient, Awareness and Engagement may use their shared
   `upper_funnel_link_ctr` group because both have the same Link CTR numerator,
   denominator, and direction.
3. Mark this fallback `provisional` and cap the final action at `KEEP_AS_TEST`.
4. Show the all-portfolio same-metric rate only as context. It never enters the prior,
   corrected score, uncertainty range, or funding decision.

The campaign being scored is always removed from its own benchmark.

```text
adjusted peer center = (peer successes + 0.5) / (peer trials + 1)
prior strength = conservative equivalent evidence allowed from peers
prior alpha = peer center x prior strength
prior beta = (1 - peer center) x prior strength
```

The prior is learned from the selected peer evidence. It is not a business target.
At least two valid compatible peers are required; otherwise the result is insufficient
evidence.

In [5]:
prior_rows = []
for _, row in campaigns.iterrows():
    same_objective_peers = all_campaigns.loc[
        all_campaigns["objective"].eq(FOCUS_OBJECTIVE)
        & all_campaigns["entity_id"].ne(row["entity_id"])
    ].copy()
    compatible_objectives = [
        name for name, candidate in registry.objectives.items()
        if contract.fallback_benchmark_group
        and candidate.fallback_benchmark_group == contract.fallback_benchmark_group
        and candidate.primary_metric == contract.primary_metric
        and candidate.primary_numerator == contract.primary_numerator
        and candidate.primary_denominator == contract.primary_denominator
        and candidate.primary_direction == contract.primary_direction
    ]
    if row["benchmark_scope"] == "same_objective":
        selected_objectives = [FOCUS_OBJECTIVE]
    elif row["benchmark_scope"] == "shared_primary_kpi_group":
        selected_objectives = compatible_objectives
    else:
        selected_objectives = []
    peers = all_campaigns.loc[
        all_campaigns["objective"].isin(selected_objectives)
        & all_campaigns["entity_id"].ne(row["entity_id"])
    ].copy()
    for frame in [same_objective_peers, peers]:
        frame["_successes"] = pd.to_numeric(
            frame[contract.primary_numerator], errors="coerce"
        )
        frame["_trials"] = pd.to_numeric(
            frame[contract.primary_denominator], errors="coerce"
        )
    same_objective_peers = same_objective_peers.loc[
        same_objective_peers["_trials"].gt(0)
        & same_objective_peers["_successes"].ge(0)
        & same_objective_peers["_successes"].le(same_objective_peers["_trials"])
    ]
    peers = peers.loc[
        peers["_trials"].gt(0)
        & peers["_successes"].ge(0)
        & peers["_successes"].le(peers["_trials"])
    ]
    item = {
        "Campaign": row["campaign_name"],
        "Own evidence": f"{row['score_successes']:.0f} / {row['score_trials']:.0f}",
        "Same-objective peers": len(same_objective_peers),
        "Selected peers": len(peers),
        "Benchmark source": row["benchmark_scope"],
        "Benchmark quality": row["benchmark_quality"],
        "Peer evidence": (
            f"{peers['_successes'].sum():.0f} / {peers['_trials'].sum():.0f}"
            if len(peers) else "None"
        ),
        "Prior center": np.nan,
        "Prior strength": np.nan,
        "Prior alpha": np.nan,
        "Prior beta": np.nan,
        "Posterior alpha": np.nan,
        "Posterior beta": np.nan,
        "Corrected rate": row["corrected_rate"],
        "Portfolio context": row["portfolio_context_benchmark"],
    }
    if len(peers) >= contract.minimum_peer_entities and row["score_trials"] > 0:
        prior = fit_beta_prior(peers["_successes"], peers["_trials"])
        failures = row["score_trials"] - row["score_successes"]
        item.update({
            "Prior center": prior.mean,
            "Prior strength": prior.strength,
            "Prior alpha": prior.alpha,
            "Prior beta": prior.beta,
            "Posterior alpha": prior.alpha + row["score_successes"],
            "Posterior beta": prior.beta + failures,
        })
    prior_rows.append(item)

prior_table = pd.DataFrame(prior_rows)
display(prior_table.style.format({
    "Prior center": "{:.2%}", "Prior strength": "{:.2f}",
    "Prior alpha": "{:.2f}", "Prior beta": "{:.2f}",
    "Posterior alpha": "{:.2f}", "Posterior beta": "{:.2f}",
    "Corrected rate": "{:.2%}", "Portfolio context": "{:.2%}",
}, na_rep="Insufficient peers"))

,Campaign,Own evidence,Same-objective peers,Selected peers,Benchmark source,Benchmark quality,Peer evidence,Prior center,Prior strength,Prior alpha,Prior beta,Posterior alpha,Posterior beta,Corrected rate,Portfolio context
0,Always-On Premium Acquisition,52 / 103,5,5,same_objective,decision_grade,195 / 308,63.27%,58.65,37.10,21.54,89.10,72.54,55.12%,59.22%
1,Ramadan Suhoor Specials,41 / 62,5,5,same_objective,decision_grade,206 / 349,59.00%,39.97,23.58,16.39,64.58,37.39,63.33%,56.63%
2,Ramadan Iftar Premium Bundles,42 / 76,5,5,same_objective,decision_grade,205 / 335,61.16%,38.15,23.33,14.82,65.33,48.82,57.23%,58.02%
3,Eid Gifting Premium,28 / 46,5,5,same_objective,decision_grade,219 / 365,59.97%,33.95,20.36,13.59,48.36,31.59,60.49%,57.37%
4,Summer Premium Launch,64 / 90,5,5,same_objective,decision_grade,183 / 321,56.99%,64.20,36.59,27.61,100.59,53.61,65.23%,55.15%
5,Mid-Year Sale,20 / 34,5,5,same_objective,decision_grade,227 / 377,60.19%,34.27,20.63,13.64,40.63,27.64,59.51%,57.58%


## 6. Raw Rate, Corrected Rate, Benchmark, And Range

The corrected rate is a weighted compromise between peer evidence and the campaign's
own evidence. Small samples move more toward the selected peer center; large samples
retain more of their raw result. The horizontal line is the campaign's 95% plausible
score range. The portfolio-context point is displayed for orientation only.

In [6]:
score_chart_data = campaigns.dropna(subset=["corrected_rate"]).copy()
if score_chart_data.empty:
    display(Markdown(
        "**No corrected campaign scores:** neither the same objective nor a configured "
        "compatible fallback group has two valid peers."
    ))
else:
    y = alt.Y("campaign_name:N", sort="-x", title=None)
    ranges = alt.Chart(score_chart_data).mark_rule(strokeWidth=4, color="#8b95a5").encode(
        y=y,
        x=alt.X("range_low:Q", title="Primary rate", axis=alt.Axis(format=".0%")),
        x2="range_high:Q",
        tooltip=[
            alt.Tooltip("campaign_name:N", title="Campaign"),
            alt.Tooltip("range_low:Q", title="95% low", format=".2%"),
            alt.Tooltip("range_high:Q", title="95% high", format=".2%"),
        ],
    )
    points = score_chart_data[[
        "campaign_name", "raw_rate", "corrected_rate", "benchmark",
        "portfolio_context_benchmark"
    ]].melt("campaign_name", var_name="Estimate", value_name="Rate")
    point_layer = alt.Chart(points).mark_point(filled=True, size=100).encode(
        y=y,
        x=alt.X("Rate:Q", axis=alt.Axis(format=".0%")),
        color=alt.Color(
            "Estimate:N",
            scale=alt.Scale(
                domain=[
                    "raw_rate", "corrected_rate", "benchmark",
                    "portfolio_context_benchmark"
                ],
                range=["#2f6f8f", "#16815d", "#d68c16", "#7b61a8"],
            ),
            legend=alt.Legend(orient="top", title=None),
        ),
        shape=alt.Shape("Estimate:N", legend=None),
        tooltip=[
            alt.Tooltip("campaign_name:N", title="Campaign"),
            alt.Tooltip("Estimate:N"),
            alt.Tooltip("Rate:Q", format=".2%"),
        ],
    )
    display((ranges + point_layer).properties(
        width=760, height=max(140, len(score_chart_data) * 52)
    ))

alt.LayerChart(...)

## 7. Probability Better And The Funding Decision

For each scored campaign, the engine creates 5,000 plausible campaign rates and 5,000
plausible peer rates. `probability_better` is the fraction where favorable lift is
above zero.

```text
higher-is-better lift = campaign draw - benchmark draw
lower-is-better lift  = benchmark draw - campaign draw

SCALE: lift low > 0
KILL:  lift high < 0
HOLD:  lift range crosses 0
```

Probability better supports interpretation and the two-factor budget priority. The complete lift
range, not the probability alone, controls the statistical decision. A provisional
fallback can still calculate a statistical direction, but its final funding action is
always capped at `KEEP_AS_TEST`.

In [7]:
decision_table = campaigns[[
    "campaign_name", "raw_rate", "corrected_rate", "benchmark",
    "benchmark_scope", "benchmark_quality", "portfolio_context_benchmark",
    "expected_lift", "lift_low", "lift_high", "probability_better",
    "statistical_decision", "recommended_action",
]].rename(columns={
    "campaign_name": "Campaign", "raw_rate": "Raw rate",
    "corrected_rate": "Corrected rate", "benchmark": "Benchmark",
    "benchmark_scope": "Benchmark source",
    "benchmark_quality": "Benchmark quality",
    "portfolio_context_benchmark": "Portfolio context",
    "expected_lift": "Expected lift", "lift_low": "Lift low",
    "lift_high": "Lift high", "probability_better": "Probability better",
    "statistical_decision": "Statistical decision",
    "recommended_action": "Final action",
})
display(decision_table.style.format({
    "Raw rate": "{:.2%}", "Corrected rate": "{:.2%}",
    "Benchmark": "{:.2%}", "Expected lift": "{:+.2%}",
    "Lift low": "{:+.2%}", "Lift high": "{:+.2%}",
    "Probability better": "{:.2%}", "Portfolio context": "{:.2%}",
}, na_rep="Not available"))

lift_data = campaigns.dropna(subset=["lift_low", "lift_high"]).copy()
if not lift_data.empty:
    lift_ranges = alt.Chart(lift_data).mark_rule(strokeWidth=5).encode(
        y=alt.Y("campaign_name:N", sort="-x", title=None),
        x=alt.X("lift_low:Q", title="Favorable lift vs peer", axis=alt.Axis(format="+.0%")),
        x2="lift_high:Q",
        color=alt.Color(
            "statistical_decision:N",
            scale=alt.Scale(
                domain=list(DECISION_COLORS), range=list(DECISION_COLORS.values())
            ),
            legend=alt.Legend(orient="top", title="Decision"),
        ),
        tooltip=[
            alt.Tooltip("campaign_name:N", title="Campaign"),
            alt.Tooltip("lift_low:Q", format="+.2%"),
            alt.Tooltip("expected_lift:Q", format="+.2%"),
            alt.Tooltip("lift_high:Q", format="+.2%"),
            alt.Tooltip("probability_better:Q", format=".2%"),
        ],
    )
    expected = alt.Chart(lift_data).mark_point(
        filled=True, color="#20262e", size=90
    ).encode(y=alt.Y("campaign_name:N", sort="-x"), x="expected_lift:Q")
    zero = alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(
        strokeDash=[5, 4], color="#20262e"
    ).encode(x="zero:Q")
    display((lift_ranges + expected + zero).properties(
        width=760, height=max(140, len(lift_data) * 52)
    ))

,Campaign,Raw rate,Corrected rate,Benchmark,Benchmark source,Benchmark quality,Portfolio context,Expected lift,Lift low,Lift high,Probability better,Statistical decision,Final action
0,Always-On Premium Acquisition,50.49%,55.12%,63.27%,same_objective,decision_grade,59.22%,-8.27%,-22.56%,+6.43%,13.24%,hold,keep_as_test
4,Ramadan Suhoor Specials,66.13%,63.33%,59.00%,same_objective,decision_grade,56.63%,+4.32%,-13.41%,+22.08%,68.92%,hold,keep_as_test
5,Ramadan Iftar Premium Bundles,55.26%,57.23%,61.16%,same_objective,decision_grade,58.02%,-3.92%,-21.34%,+13.86%,32.78%,hold,keep_as_test
6,Eid Gifting Premium,60.87%,60.49%,59.97%,same_objective,decision_grade,57.37%,+0.43%,-18.70%,+20.41%,51.10%,hold,keep_as_test
8,Summer Premium Launch,71.11%,65.23%,56.99%,same_objective,decision_grade,55.15%,+8.27%,-6.01%,+22.54%,87.22%,hold,keep_as_test
10,Mid-Year Sale,58.82%,59.51%,60.19%,same_objective,decision_grade,57.58%,-0.45%,-20.08%,+19.53%,47.74%,hold,keep_as_test


alt.LayerChart(...)

## 8. Efficiency Is A Separate Gate

The primary score answers whether the objective outcome was achieved. Efficiency asks
what it cost, or what revenue return it produced. The peer benchmark is the median
efficiency of other campaigns with this objective.

The Awareness-plus-Engagement fallback does **not** combine efficiency metrics:
Awareness keeps CPM, while Engagement keeps CPC. Their shared Link CTR is useful for
stabilizing response evidence, but their costs answer different business questions.

A statistical `SCALE` is promoted to final `SCALE` only when efficiency is at least as
good as its peer benchmark. Efficiency is not mixed into the primary score.

In [8]:
efficiency_table = campaigns[[
    "campaign_name", "efficiency_metric", "efficiency_direction",
    "efficiency_value", "efficiency_benchmark", "efficiency_peer_count",
    "efficiency_comparison",
]].rename(columns={
    "campaign_name": "Campaign", "efficiency_metric": "Metric",
    "efficiency_direction": "Better direction", "efficiency_value": "Value",
    "efficiency_benchmark": "Peer median", "efficiency_peer_count": "Peers",
    "efficiency_comparison": "Comparison",
})
display(efficiency_table.style.format({
    "Value": "{:,.3f}", "Peer median": "{:,.3f}"
}, na_rep="Not available"))

efficiency_long = efficiency_table.melt(
    id_vars=["Campaign", "Metric", "Better direction", "Comparison"],
    value_vars=["Value", "Peer median"],
    var_name="Measure", value_name="Efficiency",
).dropna(subset=["Efficiency"])
if not efficiency_long.empty:
    efficiency_chart = alt.Chart(efficiency_long).mark_bar().encode(
        y=alt.Y("Campaign:N", title=None),
        x=alt.X("Efficiency:Q", title=contract.efficiency_metric.replace("_", " ").title()),
        yOffset="Measure:N",
        color=alt.Color(
            "Measure:N",
            scale=alt.Scale(domain=["Value", "Peer median"], range=["#2f6f8f", "#d68c16"]),
            legend=alt.Legend(orient="top", title=None),
        ),
        tooltip=["Campaign", "Metric", "Better direction", "Measure", "Efficiency"],
    ).properties(width=760, height=max(140, len(campaigns) * 58))
    display(efficiency_chart)

,Campaign,Metric,Better direction,Value,Peer median,Peers,Comparison
0,Always-On Premium Acquisition,net_roas,higher,0.468,1.335,5,worse_than_peer
4,Ramadan Suhoor Specials,net_roas,higher,1.177,1.335,5,worse_than_peer
5,Ramadan Iftar Premium Bundles,net_roas,higher,1.349,1.177,5,better_than_peer
6,Eid Gifting Premium,net_roas,higher,4.340,1.177,5,better_than_peer
8,Summer Premium Launch,net_roas,higher,1.028,1.335,5,worse_than_peer
10,Mid-Year Sale,net_roas,higher,1.335,1.177,5,better_than_peer


alt.Chart(...)

## 9. Final Campaign Action And Budget

The statistical decision and evidence status create reporting labels:

| Statistical evidence | Efficiency | Final action |
|---|---|---|
| SCALE | Better or equal to peer | SCALE |
| SCALE | Worse or unavailable | KEEP_AS_TEST |
| HOLD | Any | KEEP_AS_TEST |
| KILL | Any | DO_NOT_FUND |
| Too little evidence | Any | INSUFFICIENT_EVIDENCE |

For a `provisional` shared-Link-CTR benchmark, any assessable statistical result is
capped at `KEEP_AS_TEST`. It cannot trigger `SCALE` or `DO_NOT_FUND`.

For this POC, those labels do not gate budget. Previous-cycle spend shares preserve
each objective envelope, then all campaigns with both components share the full
envelope using `probability_better x semantic range_low`. Missing components receive
zero; an objective with no valid priorities remains unallocated.

In [9]:
recommendation_rows = []
for _, row in campaigns.iterrows():
    allocation = allocations[str(row["campaign_id"])]
    recommendation_rows.append({
        "Campaign": row["campaign_name"],
        "Benchmark source": row["benchmark_scope"],
        "Benchmark quality": row["benchmark_quality"],
        "Statistical decision": row["statistical_decision"],
        "Efficiency": row["efficiency_comparison"],
        "Final action": row["recommended_action"],
        "Budget pool": allocation.budget_pool,
        "Primary probability": allocation.primary_probability_component,
        "Quality lower bound": allocation.semantic_priority,
        "Priority product": allocation.allocation_priority,
        "Normalized weight": allocation.allocation_weight,
        "Budget units": allocation.recommended_budget_units,
        "Previous-spend scenario": allocation.previous_spend_budget_units,
        "Allocation basis": allocation.allocation_basis,
        "Portfolio share": allocation.recommended_budget_share,
        "Reason codes": ", ".join(allocation.reason_codes),
    })
recommendation_table = pd.DataFrame(recommendation_rows)
display(recommendation_table.style.format({
    "Budget units": "{:.2f}", "Portfolio share": "{:.2%}",
    "Primary probability": "{:.2%}", "Quality lower bound": "{:.2%}",
    "Priority product": "{:.4f}", "Normalized weight": "{:.2%}"
}))

budget_data = recommendation_table[recommendation_table["Budget units"].gt(0)]
if not budget_data.empty:
    budget_chart = alt.Chart(budget_data).mark_bar().encode(
        y=alt.Y("Campaign:N", sort="-x", title=None),
        x=alt.X("Budget units:Q", title="Recommended next-cycle budget units"),
        color=alt.value("#16815d"),
        tooltip=["Campaign", "Final action", "Primary probability",
                 "Quality lower bound", "Priority product",
                 alt.Tooltip("Budget units:Q", format=".2f")],
    ).properties(width=760, height=max(130, len(budget_data) * 48))
    display(budget_chart)
else:
    display(Markdown("**No budget is assigned to this objective under the current evidence rules.**"))

,Campaign,Benchmark source,Benchmark quality,Statistical decision,Efficiency,Final action,Budget pool,Primary probability,Quality lower bound,Priority product,Normalized weight,Budget units,Previous-spend scenario,Allocation basis,Portfolio share,Reason codes
0,Always-On Premium Acquisition,same_objective,decision_grade,hold,worse_than_peer,keep_as_test,score_based,13.24%,54.89%,0.0727,4.01%,3.30,34.996964,primary_probability_x_semantic_lower_bound,3.30%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"
1,Ramadan Suhoor Specials,same_objective,decision_grade,hold,worse_than_peer,keep_as_test,score_based,68.92%,62.63%,0.4317,23.83%,19.63,8.027695,primary_probability_x_semantic_lower_bound,19.63%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"
2,Ramadan Iftar Premium Bundles,same_objective,decision_grade,hold,better_than_peer,keep_as_test,score_based,32.78%,59.95%,0.1965,10.85%,8.94,14.193104,primary_probability_x_semantic_lower_bound,8.94%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"
3,Eid Gifting Premium,same_objective,decision_grade,hold,better_than_peer,keep_as_test,score_based,51.10%,56.63%,0.2894,15.98%,13.16,3.722889,primary_probability_x_semantic_lower_bound,13.16%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"
4,Summer Premium Launch,same_objective,decision_grade,hold,worse_than_peer,keep_as_test,score_based,87.22%,64.19%,0.5599,30.91%,25.46,17.051992,primary_probability_x_semantic_lower_bound,25.46%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"
5,Mid-Year Sale,same_objective,decision_grade,hold,better_than_peer,keep_as_test,score_based,47.74%,54.72%,0.2612,14.42%,11.88,4.363310,primary_probability_x_semantic_lower_bound,11.88%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"


alt.Chart(...)

## 10. Named Tests For KEEP_AS_TEST Campaigns

The score-based budget is independent of action labels. When a funded campaign is
labelled `KEEP_AS_TEST`, the report also creates a specific hypothesis with a success
rule, failure rule, and stop rule for learning during the next cycle.

In [10]:
if objective_tests:
    tests_table = pd.DataFrame([
        {
            "Campaign": item.campaign_name,
            "Budget units": item.assigned_budget_units,
            "Hypothesis": item.hypothesis,
            "Primary metric": item.primary_metric,
            "Success rule": item.success_rule,
            "Failure rule": item.failure_rule,
            "Stop rule": item.stop_rule,
        }
        for item in objective_tests
    ])
    display(tests_table.style.format({"Budget units": "{:.2f}"}))
else:
    display(Markdown(
        "No funded KEEP_AS_TEST campaign is present for this objective."
    ))

,Campaign,Budget units,Hypothesis,Primary metric,Success rule,Failure rule,Stop rule
0,Always-On Premium Acquisition,3.30,"Clarify the offer and address the observed price barrier in a named variant against the current message. Hypothesis: this increases checkout readiness and improves customer delivered rate in the next cycle. The observed association is a hypothesis, not an established cause.",customer_delivered_rate,"At cycle end, check the primary favorable-lift range, efficiency, and the separate semantic range. A provisional Link CTR benchmark remains capped at KEEP_AS_TEST.","At cycle end, DO_NOT_FUND requires a decision-grade primary lift upper bound below zero with sufficient evidence. Semantic quality alone does not determine the final action.","Stop when the assigned budget is spent or the next cycle ends, whichever comes first; wait for outcomes to mature before scoring."
1,Ramadan Suhoor Specials,19.63,"Clarify the offer and address the observed delivery barrier in a named variant against the current message. Hypothesis: this increases checkout readiness and improves customer delivered rate in the next cycle. The observed association is a hypothesis, not an established cause.",customer_delivered_rate,"At cycle end, check the primary favorable-lift range, efficiency, and the separate semantic range. A provisional Link CTR benchmark remains capped at KEEP_AS_TEST.","At cycle end, DO_NOT_FUND requires a decision-grade primary lift upper bound below zero with sufficient evidence. Semantic quality alone does not determine the final action.","Stop when the assigned budget is spent or the next cycle ends, whichever comes first; wait for outcomes to mature before scoring."
2,Ramadan Iftar Premium Bundles,8.94,"Clarify the offer and address the observed timing barrier in a named variant against the current message. Hypothesis: this increases checkout readiness and improves customer delivered rate in the next cycle. The observed association is a hypothesis, not an established cause.",customer_delivered_rate,"At cycle end, check the primary favorable-lift range, efficiency, and the separate semantic range. A provisional Link CTR benchmark remains capped at KEEP_AS_TEST.","At cycle end, DO_NOT_FUND requires a decision-grade primary lift upper bound below zero with sufficient evidence. Semantic quality alone does not determine the final action.","Stop when the assigned budget is spent or the next cycle ends, whichever comes first; wait for outcomes to mature before scoring."
3,Eid Gifting Premium,13.16,"Clarify the offer and address the observed product_quality barrier in a named variant against the current message. Hypothesis: this increases checkout readiness and improves customer delivered rate in the next cycle. The observed association is a hypothesis, not an established cause.",customer_delivered_rate,"At cycle end, check the primary favorable-lift range, efficiency, and the separate semantic range. A provisional Link CTR benchmark remains capped at KEEP_AS_TEST.","At cycle end, DO_NOT_FUND requires a decision-grade primary lift upper bound below zero with sufficient evidence. Semantic quality alone does not determine the final action.","Stop when the assigned budget is spent or the next cycle ends, whichever comes first; wait for outcomes to mature before scoring."
4,Summer Premium Launch,25.46,"Clarify the offer and address the observed price barrier in a named variant against the current message. Hypothesis: this increases checkout readiness and improves customer delivered rate in the next cycle. The observed association is a hypothesis, not an established cause.",customer_delivered_rate,"At cycle end, check the primary favorable-lift range, efficiency, and the separate semantic range. A provisional Link CTR benchmark remains capped at KEEP_AS_TEST.","At cycle end, DO_NOT_FUND requires a decision-grade primary lift upper bound below zero with sufficient evidence. Semantic quality alone does not determine the final act

## 11. Adset, Ad, Creative, And Audience Evidence

The same objective KPI and benchmark hierarchy are applied at every level.
Campaign and adset are normal budget-control layers. Ads are run/test/stop candidates.
Creative and audience views identify reusable patterns; their budgets are not added
separately.

A larger corrected score is a leader to investigate, not automatically a winner. The
range-based decision and final action remain authoritative.

In [11]:
child_rows = []
for level in ["adset", "ad", "creative", "audience"]:
    frame = result.scorecards[level].loc[
        result.scorecards[level]["objective"].eq(FOCUS_OBJECTIVE)
    ].copy()
    for _, row in frame.iterrows():
        child_rows.append({
            "Level": level,
            "Campaign": row["campaign_name"],
            "Entity": row["entity_name"],
            "Successes": row["score_successes"],
            "Trials": row["score_trials"],
            "Raw rate": row["raw_rate"],
            "Corrected rate": row["corrected_rate"],
            "Lift low": row["lift_low"],
            "Lift high": row["lift_high"],
            "Probability better": row["probability_better"],
            "Benchmark source": row["benchmark_scope"],
            "Benchmark quality": row["benchmark_quality"],
            "Statistical decision": row["statistical_decision"],
            "Final action": row["recommended_action"],
            "Spend": row["spend"],
        })
child_summary = pd.DataFrame(child_rows)
display(pd.crosstab(
    [child_summary["Level"]], child_summary["Final action"], margins=True
))

for level in ["adset", "ad", "creative", "audience"]:
    subset = child_summary[child_summary["Level"].eq(level)].copy()
    subset = subset.sort_values(
        ["Probability better", "Trials"], ascending=[False, False], na_position="last"
    ).head(12)
    display(Markdown(f"### {level.title()} leaders and test candidates"))
    display(subset.style.format({
        "Raw rate": "{:.2%}", "Corrected rate": "{:.2%}",
        "Lift low": "{:+.2%}", "Lift high": "{:+.2%}",
        "Probability better": "{:.2%}", "Spend": "{:,.2f}",
    }, na_rep="Not available"))

Final action,insufficient_evidence,keep_as_test,All
Level,,,
ad,6,22,28
adset,1,15,16
audience,1,13,14
creative,4,17,21
All,12,67,79


### Adset leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
11,adset,Summer Premium Launch,1% Lookalike Cycle 2 Ramadan high-LTV,32.000000,37.000000,86.49%,78.28%,-5.16%,+48.59%,93.76%,same_objective,decision_grade,hold,keep_as_test,"27,475.86"
3,adset,Ramadan Suhoor Specials,Ramadan Cairo women 25-50,31.000000,40.000000,77.50%,73.04%,-13.34%,+44.19%,82.96%,same_objective,decision_grade,hold,keep_as_test,"17,438.67"
10,adset,Summer Premium Launch,Cairo+Alex high-income summer launch,21.000000,29.000000,72.41%,68.72%,-20.03%,+41.72%,71.40%,same_objective,decision_grade,hold,keep_as_test,"23,697.20"
7,adset,Ramadan Iftar Premium Bundles,Retargeting cart abandoners 30d,12.000000,17.000000,70.59%,66.29%,-24.64%,+40.18%,64.96%,same_objective,decision_grade,hold,keep_as_test,"11,346.77"
14,adset,Mid-Year Sale,Retargeting all engagers sale,4.000000,5.000000,80.00%,65.63%,-28.66%,+39.77%,62.26%,same_objective,decision_grade,hold,insufficient_evidence,"4,773.88"
8,adset,Eid Gifting Premium,Gift-buyer general Eid,22.000000,34.000000,64.71%,63.51%,-25.84%,+35.77%,58.66%,same_objective,decision_grade,hold,keep_as_test,"9,516.19"
1,adset,Always-On Premium Acquisition,1% Lookalike all-delivered customers,26.000000,42.000000,61.90%,61.50%,-27.68%,+34.29%,52.00%,same_objective,decision_grade,hold,keep_as_test,"66,428.96"
15,adset,Mid-Year Sale,1% Lookalike sale buyers historical,6.000000,10.000000,60.00%,60.04%,-33.75%,+35.72%,49.34%,same_objective,decision_grade,hold,keep_as_test,"5,559.02"
6,adset,Ramadan Iftar Premium Bundles,1% Lookalike premium customers Ramadan,13.000000,24.000000,54.17%,56.08%,-35.29%,+29.26%,40.18%,same_objective,decision_grade,hold,keep_as_test,"17,706.62"
2,adset,Always-On Premium Acquisition,3% Lookalike scale,9.000000,17.000000,52.94%,55.80%,-37.29%,+28.83%,37.76%,same_objective,decision_grade,hold,keep_as_test,"26,251.65"


### Ad leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
37,ad,Summer Premium Launch,Summer 1% LAL high-LTV refresh premium scale,20.000000,23.000000,86.96%,80.45%,-12.43%,+60.76%,86.48%,same_objective,decision_grade,hold,keep_as_test,"13,453.24"
22,ad,Ramadan Suhoor Specials,Ramadan Cairo women creative A,14.000000,16.000000,87.50%,78.97%,-15.66%,+58.82%,85.02%,same_objective,decision_grade,hold,keep_as_test,"7,810.11"
36,ad,Summer Premium Launch,Summer 1% LAL high-LTV initial,12.000000,14.000000,85.71%,77.11%,-19.43%,+57.87%,80.06%,same_objective,decision_grade,hold,keep_as_test,"14,022.62"
35,ad,Summer Premium Launch,Summer launch new arrivals refresh,16.000000,21.000000,76.19%,72.26%,-22.47%,+53.42%,73.20%,same_objective,decision_grade,hold,keep_as_test,"11,813.36"
23,ad,Ramadan Suhoor Specials,Ramadan Cairo women creative B refresh,17.000000,24.000000,70.83%,68.49%,-25.78%,+49.84%,66.86%,same_objective,decision_grade,hold,keep_as_test,"9,628.56"
32,ad,Eid Gifting Premium,Eid gift hostess refresh,13.000000,18.000000,72.22%,68.96%,-28.38%,+51.80%,65.12%,same_objective,decision_grade,hold,keep_as_test,"3,228.72"
42,ad,Mid-Year Sale,Retargeting comeback,4.000000,5.000000,80.00%,68.63%,-33.50%,+52.72%,64.90%,same_objective,decision_grade,hold,insufficient_evidence,"4,773.88"
29,ad,Ramadan Iftar Premium Bundles,Iftar retargeting scarcity,12.000000,17.000000,70.59%,67.66%,-30.76%,+50.53%,62.46%,same_objective,decision_grade,hold,keep_as_test,"11,346.77"
20,ad,Always-On Premium Acquisition,1% LAL acquisition refresh,11.000000,16.000000,68.75%,66.23%,-31.79%,+49.22%,61.04%,same_objective,decision_grade,hold,keep_as_test,"32,011.40"
39,ad,Summer Premium Launch,Summer 3% LAL refresh,10.000000,16.000000,62.50%,61.77%,-36.00%,+44.76%,52.20%,same_objective,decision_grade,hold,keep_as_test,"8,561.97"


### Creative leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
59,creative,Summer Premium Launch,Lookalike scale premium quality,20.000000,23.000000,86.96%,80.86%,-12.80%,+62.02%,87.48%,same_objective,decision_grade,hold,keep_as_test,"13,453.24"
49,creative,Ramadan Suhoor Specials,Suhoor essentials Ramadan,20.000000,26.000000,76.92%,73.68%,-21.22%,+56.28%,75.44%,same_objective,decision_grade,hold,keep_as_test,"14,686.75"
57,creative,Summer Premium Launch,Summer Premium Launch hero,17.000000,22.000000,77.27%,73.52%,-22.86%,+56.36%,73.04%,same_objective,decision_grade,hold,keep_as_test,"25,906.46"
58,creative,Summer Premium Launch,Summer Premium Launch new arrivals,16.000000,21.000000,76.19%,72.57%,-23.46%,+55.44%,71.86%,same_objective,decision_grade,hold,keep_as_test,"11,813.36"
56,creative,Eid Gifting Premium,Eid gifting hostess,13.000000,18.000000,72.22%,69.23%,-29.82%,+52.29%,64.62%,same_objective,decision_grade,hold,keep_as_test,"3,228.72"
64,creative,Mid-Year Sale,Mid-year retargeting comeback,4.000000,5.000000,80.00%,69.13%,-34.43%,+54.99%,64.36%,same_objective,decision_grade,hold,insufficient_evidence,"4,773.88"
50,creative,Ramadan Suhoor Specials,Ramadan tradition modernized health,17.000000,24.000000,70.83%,68.69%,-28.81%,+52.25%,63.74%,same_objective,decision_grade,hold,keep_as_test,"9,628.56"
53,creative,Ramadan Iftar Premium Bundles,Iftar bundle scarcity,20.000000,30.000000,66.67%,65.57%,-29.88%,+48.33%,58.78%,same_objective,decision_grade,hold,keep_as_test,"19,563.58"
61,creative,Summer Premium Launch,Always-on premium spring refresh,10.000000,16.000000,62.50%,61.84%,-38.04%,+46.05%,51.76%,same_objective,decision_grade,hold,keep_as_test,"8,561.97"
45,creative,Always-On Premium Acquisition,Always-on premium spring refresh,1.000000,2.000000,50.00%,57.23%,-51.87%,+50.41%,44.92%,same_objective,decision_grade,hold,insufficient_evidence,"16,750.42"


### Audience leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
67,audience,Ramadan Suhoor Specials,Demographic,31.000000,40.000000,77.50%,72.12%,-11.27%,+41.14%,84.78%,same_objective,decision_grade,hold,keep_as_test,"17,438.67"
75,audience,Summer Premium Launch,Lookalike,43.000000,61.000000,70.49%,68.24%,-16.16%,+37.83%,75.82%,same_objective,decision_grade,hold,keep_as_test,"44,898.64"
74,audience,Summer Premium Launch,Demographic,21.000000,29.000000,72.41%,68.09%,-17.34%,+38.03%,71.18%,same_objective,decision_grade,hold,keep_as_test,"23,697.20"
71,audience,Ramadan Iftar Premium Bundles,Retargeting,12.000000,17.000000,70.59%,65.71%,-23.10%,+36.24%,64.94%,same_objective,decision_grade,hold,keep_as_test,"11,346.77"
77,audience,Mid-Year Sale,Retargeting,4.000000,5.000000,80.00%,64.58%,-25.32%,+35.66%,60.88%,same_objective,decision_grade,hold,insufficient_evidence,"4,773.88"
72,audience,Eid Gifting Premium,Broad,22.000000,34.000000,64.71%,63.31%,-23.52%,+33.84%,58.36%,same_objective,decision_grade,hold,keep_as_test,"9,516.19"
78,audience,Mid-Year Sale,Lookalike,6.000000,10.000000,60.00%,60.04%,-31.53%,+32.66%,48.82%,same_objective,decision_grade,hold,keep_as_test,"5,559.02"
66,audience,Always-On Premium Acquisition,Lookalike,35.000000,59.000000,59.32%,59.48%,-28.07%,+28.84%,46.18%,same_objective,decision_grade,hold,keep_as_test,"92,680.61"
70,audience,Ramadan Iftar Premium Bundles,Lookalike,13.000000,24.000000,54.17%,56.37%,-32.62%,+26.11%,38.38%,same_objective,decision_grade,hold,keep_as_test,"17,706.62"
73,audience,Eid Gifting Premium,Lookalike,6.000000,12.000000,50.00%,55.50%,-35.27%,+27.28%,38.16%,same_objective,decision_grade,hold,keep_as_test,"5,460.05"


In [12]:
child_chart_data = child_summary.dropna(subset=["Lift low", "Lift high"]).copy()
child_chart_data = child_chart_data.sort_values(
    "Probability better", ascending=False
).groupby("Level", as_index=False).head(10)
for level in ["adset", "ad", "creative", "audience"]:
    level_data = child_chart_data[child_chart_data["Level"].eq(level)]
    if level_data.empty:
        continue
    base = alt.Chart(level_data)
    child_chart = base.mark_rule(strokeWidth=4).encode(
        y=alt.Y("Entity:N", sort="-x", title=None),
        x=alt.X("Lift low:Q", title="Favorable lift vs selected peer", axis=alt.Axis(format="+.0%")),
        x2="Lift high:Q",
        color=alt.Color(
            "Statistical decision:N",
            scale=alt.Scale(domain=list(DECISION_COLORS), range=list(DECISION_COLORS.values())),
            legend=alt.Legend(orient="top"),
        ),
        tooltip=[
            "Level", "Campaign", "Entity", "Successes", "Trials",
            alt.Tooltip("Probability better:Q", format=".2%"),
            "Final action",
        ],
    )
    zero = base.mark_rule(color="#20262e", strokeDash=[5, 4]).encode(
        x=alt.datum(0)
    )
    display(Markdown(f"### {level.title()} favorable-lift ranges"))
    display((child_chart + zero).properties(
        width=760, height=max(180, len(level_data) * 38)
    ))

### Adset favorable-lift ranges

alt.LayerChart(...)

### Ad favorable-lift ranges

alt.LayerChart(...)

### Creative favorable-lift ranges

alt.LayerChart(...)

### Audience favorable-lift ranges

alt.LayerChart(...)

## 12. What Conversation Signals Contribute

Intent, delivery readiness, agreement, barriers, value drivers, and agent behavior help explain observed delivery performance. They cannot replace delivered outcomes.

The semantic artifact is joined back to structured outcomes only after extraction.
The extraction model did not receive revenue, outcome, customer identity, budget, or
funding decisions. `unknown` is excluded from assessable denominators rather than
converted to false.

`next_step_order_progression_rate` means an agreed next step followed by an observed
structured order. It is an order-progression proxy, not proof that every promised task
was completed.

The following sections calculate a separate objective-specific quality score. These
full-conversation labels describe historical interest and agreement; they may record
an agreement followed by cancellation. They are not predictions of later purchases.

In [13]:
diagnostic_columns = [
    "semantic_coverage_rate", "high_purchase_intent_rate",
    "price_blocking_rate", "barrier_resolution_rate",
    "ad_alignment_rate", "agent_helpful_rate",
    "next_step_agreement_rate", "next_step_order_progression_rate",
]
semantic_table = campaigns[[
    "campaign_name", "semantic_conversations", *diagnostic_columns,
    "top_customer_need", "top_barrier", "top_value_driver",
]].rename(columns={"campaign_name": "Campaign"})
display(semantic_table.style.format(
    {column: "{:.2%}" for column in diagnostic_columns},
    na_rep="Unknown / not assessable",
))

heatmap = semantic_table[["Campaign", *diagnostic_columns]].melt(
    "Campaign", var_name="Signal", value_name="Rate"
).dropna(subset=["Rate"])
if not heatmap.empty:
    base = alt.Chart(heatmap).encode(
        x=alt.X("Signal:N", title=None, axis=alt.Axis(labelAngle=-35)),
        y=alt.Y("Campaign:N", title=None),
    )
    rectangles = base.mark_rect().encode(
        color=alt.Color(
            "Rate:Q", scale=alt.Scale(domain=[0, 0.5, 1], range=["#c44536", "#f2cf63", "#16815d"]),
            legend=alt.Legend(format=".0%", orient="top"),
        ),
        tooltip=["Campaign", "Signal", alt.Tooltip("Rate:Q", format=".2%")],
    )
    labels = base.mark_text(size=11).encode(
        text=alt.Text("Rate:Q", format=".0%"),
        color=alt.condition("datum.Rate < 0.25 || datum.Rate > 0.78", alt.value("white"), alt.value("#20262e")),
    )
    display((rectangles + labels).properties(width=760, height=max(110, len(campaigns) * 48)))

,Campaign,semantic_conversations,semantic_coverage_rate,high_purchase_intent_rate,price_blocking_rate,barrier_resolution_rate,ad_alignment_rate,agent_helpful_rate,next_step_agreement_rate,next_step_order_progression_rate,top_customer_need,top_barrier,top_value_driver
0,Always-On Premium Acquisition,111,100.00%,65.14%,1.52%,41.67%,51.35%,100.00%,61.11%,89.39%,looking for genuine saffron,price,price
4,Ramadan Suhoor Specials,63,98.44%,80.95%,0.00%,54.55%,50.79%,98.39%,79.37%,96.00%,Buy sugar-free basbousa (taste similar) and one healthy konafa; arrange delivery,delivery,delivery_speed
5,Ramadan Iftar Premium Bundles,84,100.00%,85.71%,5.13%,36.59%,46.43%,98.78%,71.08%,98.31%,طلب Iftar Bundle للعزومة غداً مع توصيل قبل المغرب,timing,price
6,Eid Gifting Premium,47,100.00%,86.96%,3.45%,40.91%,67.39%,97.83%,78.26%,100.00%,شراء Stuffed Dried Dates Premium للعيد,product_quality,gifting
8,Summer Premium Launch,97,100.00%,79.38%,1.89%,67.86%,48.45%,98.96%,79.38%,94.81%,Replace or compensate for a jar that appeared cracked on delivery (orange blossom honey).,price,gifting
10,Mid-Year Sale,33,97.06%,87.88%,7.14%,28.57%,69.70%,100.00%,72.73%,91.67%,"Place an order (3 Truffles, 1 Saffron, 2 Coffee Lover Boxes) and later return one sealed Coffee Lover Box",product_fit,price


alt.LayerChart(...)

### 12.1 Define Conversation Quality Before Counting

Each objective has one explicit rule. These definitions are MVP assumptions that
need transcript review. They do not combine labels using subjective weights.
A positive evidence-bearing label must cite message indexes. All required components
must be known; an unknown component leaves the combined result unknown.

| Objective | Success definition |
|---|---|
| Awareness | Central need aligned with the ad; partial/mismatch are negative |
| Engagement | Commercial inquiry, medium/high specificity, consideration/checkout, aligned/partial ad match |
| Leads | Commercial inquiry, medium/high specificity and medium/high purchase intent |
| Sales | Commercial inquiry, complete sales agreement and accepted next step |

Commercial inquiry means purchase, product information, promotion information or
delivery information. These semantic rates describe customers who chatted. They do
not measure awareness lift or engagement among everyone who viewed an ad.

In [14]:
display(Markdown(f"**This objective:** {contract.semantic.metric}: {contract.semantic.definition}"))
objective_audit = result.semantic_evidence.loc[
    result.semantic_evidence["objective"].eq(FOCUS_OBJECTIVE)
].copy()
display(objective_audit.groupby(["entity_level", "exclusion_reason"], dropna=False).size().rename("Records").reset_index())

**This objective:** checkout_readiness: Commercial inquiry with complete sales agreement and customer-accepted next step.

,entity_level,exclusion_reason,Records
0,ad,unresolved,25
1,ad,NaN,412
2,adset,repeat_customer,1
3,adset,unresolved,25
4,adset,NaN,411
5,audience,repeat_customer,1
6,audience,unresolved,25
7,audience,NaN,411
8,campaign,repeat_customer,1
9,campaign,unresolved,25


### 12.2 Inspect Selection And Raw Evidence

Choose the earliest mature conversation per customer within each entity, using
started_at then conversation ID (missing timestamps last). Choose before checking
labels. Later records do not replace an earlier unknown or missing result.
This prevents repeat chats from multiplying a customer's weight. Open outcomes
stay out of this completed-cycle score. A customer can still appear across entities.

The local audit below connects every selection and exclusion to a conversation ID.
No transcript or customer identity is passed to the recommendation model.

In [15]:
campaign_audit = objective_audit[objective_audit["entity_level"].eq("campaign")].copy()
display(campaign_audit[[
    "campaign_id", "conversation_id", "metric", "selected",
    "exclusion_reason", "signal_available", "success"
]])
semantic_tables = {}
for level, frame in result.scorecards.items():
    focused = frame[frame["objective"].eq(FOCUS_OBJECTIVE)].reset_index(drop=True)
    semantic_tables[level] = pd.concat([
        focused[["entity_id", "entity_name", "campaign_name"]],
        pd.json_normalize(focused["semantic_score"])
    ], axis=1)
quality = semantic_tables["campaign"]
display(quality[[
    "entity_name", "eligible_customers", "successes", "trials",
    "unknown_customers", "missing_customers", "raw_rate", "evidence_status"
]].style.format({"raw_rate": "{:.2%}"}, na_rep="Unknown"))
assert (quality["trials"] + quality["unknown_customers"] + quality["missing_customers"]).equals(quality["eligible_customers"])

,campaign_id,conversation_id,metric,selected,exclusion_reason,signal_available,success
0,120209876543220001,conv_688,checkout_readiness,True,None,True,True
1,120209876543220001,conv_677,checkout_readiness,True,None,True,False
2,120209876543220001,conv_373,checkout_readiness,True,None,True,False
3,120209876543220001,conv_044,checkout_readiness,True,None,True,False
4,120209876543220001,conv_225,checkout_readiness,True,None,True,True
...,...,...,...,...,...,...,...
609,120209876543220009,conv_741,checkout_readiness,True,None,True,True
610,120209876543220009,conv_247,checkout_readiness,False,unresolved,True,False
611,120209876543220009,conv_155,checkout_readiness,True,None,True,True
613,120209876543220001,conv_652,checkout_readiness,False,unresolved,True,False


,entity_name,eligible_customers,successes,trials,unknown_customers,missing_customers,raw_rate,evidence_status
0,Always-On Premium Acquisition,103,56,98,5,0,57.14%,sufficient
1,Ramadan Suhoor Specials,62,46,61,0,1,75.41%,sufficient
2,Ramadan Iftar Premium Bundles,76,52,75,1,0,69.33%,sufficient
3,Eid Gifting Premium,46,30,45,1,0,66.67%,sufficient
4,Summer Premium Launch,90,67,89,1,0,75.28%,sufficient
5,Mid-Year Sale,34,21,33,0,1,63.64%,sufficient


### 12.3 Calculate The Corrected Quality And Range

With two valid same-objective peers, learn the prior with the same Empirical-Bayes
method used earlier. Exclude the entity itself. Semantic metrics differ across
objectives, so the shared Link CTR fallback does not apply here.

With fewer peers, use Beta(0.5, 0.5), a weak Jeffreys prior: half a success and half a
failure worth of mathematical smoothing, not real customer records. Show its range
but no peer benchmark, lift or probability-better claim.

```text
raw rate = successes / assessable customers
posterior alpha = prior alpha + successes
posterior beta = prior beta + assessable customers - successes
corrected rate = posterior alpha / (posterior alpha + posterior beta)
95% credible range = 2.5th and 97.5th percentiles of posterior draws
```

The code uses 5,000 repeatable draws. With a valid peer prior, favorable lift above
zero across the whole range means supportive; below zero means concerning; crossing
zero means neutral. Fewer than 10 assessable customers means insufficient evidence.
Ten is an MVP evidence floor, not a universal statistical guarantee.

In [16]:
for level, table in semantic_tables.items():
    display(Markdown(f"**{level.title()}: raw evidence through posterior**"))
    display(table[[
        "entity_name", "metric", "successes", "trials", "prior_source", "peer_count",
        "prior_alpha", "prior_beta", "posterior_alpha", "posterior_beta",
        "raw_rate", "corrected_rate", "range_low", "range_high",
        "benchmark", "lift_low", "lift_high", "probability_better", "status"
    ]].style.format({
        **{c: "{:.2%}" for c in ["raw_rate", "corrected_rate", "range_low", "range_high", "benchmark", "probability_better"]},
        **{c: "{:.3f}" for c in ["prior_alpha", "prior_beta", "posterior_alpha", "posterior_beta"]},
        "lift_low": "{:+.2%}", "lift_high": "{:+.2%}"
    }, na_rep="Not available"))
    available = table[table["trials"].gt(0)]
    calculated = available["posterior_alpha"] / (available["posterior_alpha"] + available["posterior_beta"])
    assert np.allclose(calculated, available["corrected_rate"])

**Campaign: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Always-On Premium Acquisition,checkout_readiness,56,98,same_objective_empirical,5,43.158,17.442,99.158,59.442,57.14%,62.52%,54.89%,69.78%,71.22%,-21.87%,+5.41%,11.22%,neutral
1,Ramadan Suhoor Specials,checkout_readiness,46,61,same_objective_empirical,5,32.097,16.225,78.097,31.225,75.41%,71.44%,62.63%,79.50%,66.42%,-10.00%,+21.05%,72.80%,neutral
2,Ramadan Iftar Premium Bundles,checkout_readiness,52,75,same_objective_empirical,5,23.413,11.308,75.413,34.308,69.33%,68.73%,59.95%,76.84%,67.43%,-15.84%,+18.69%,54.40%,neutral
3,Eid Gifting Premium,checkout_readiness,30,45,same_objective_empirical,5,23.336,11.019,53.336,26.019,66.67%,67.21%,56.63%,77.23%,67.93%,-18.77%,+17.63%,45.68%,neutral
4,Summer Premium Launch,checkout_readiness,67,89,same_objective_empirical,5,31.672,16.568,98.672,38.568,75.28%,71.90%,64.19%,79.18%,65.65%,-8.06%,+22.06%,78.86%,neutral
5,Mid-Year Sale,checkout_readiness,21,33,same_objective_empirical,5,25.452,11.891,46.452,23.891,63.64%,66.04%,54.72%,76.32%,68.16%,-20.45%,+16.56%,40.90%,neutral


**Adset: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Cairo broad acquisition,checkout_readiness,18,41,same_objective_empirical,15,16.920,7.080,34.920,30.080,43.90%,53.72%,41.13%,65.07%,70.50%,-37.42%,+5.78%,6.74%,neutral
1,1% Lookalike all-delivered customers,checkout_readiness,26,42,same_objective_empirical,15,14.376,6.619,40.376,22.619,61.90%,64.09%,52.20%,75.52%,68.47%,-25.52%,+19.46%,34.18%,neutral
2,3% Lookalike scale,checkout_readiness,12,15,same_objective_empirical,15,15.627,7.588,27.627,10.588,80.00%,72.29%,57.06%,84.86%,67.31%,-17.25%,+29.16%,66.68%,neutral
3,Ramadan Cairo women 25-50,checkout_readiness,31,39,same_objective_empirical,15,15.480,7.788,46.480,15.788,79.49%,74.64%,63.12%,84.40%,66.53%,-12.75%,+30.04%,76.40%,neutral
4,Ramadan multi-city Cairo+Alex+Mansoura+Tanta,checkout_readiness,15,22,same_objective_empirical,15,13.939,6.631,28.939,13.631,68.18%,67.98%,53.45%,81.00%,67.76%,-23.23%,+25.11%,49.76%,neutral
5,High-income Cairo+Alex Iftar,checkout_readiness,23,35,same_objective_empirical,15,14.005,6.596,37.005,18.596,65.71%,66.56%,53.55%,78.40%,67.98%,-23.55%,+23.14%,44.44%,neutral
6,1% Lookalike premium customers Ramadan,checkout_readiness,15,23,same_objective_empirical,15,14.038,6.624,29.038,14.624,65.22%,66.51%,52.03%,79.53%,67.94%,-24.47%,+23.21%,44.66%,neutral
7,Retargeting cart abandoners 30d,checkout_readiness,14,17,same_objective_empirical,15,16.502,8.076,30.502,11.076,82.35%,73.36%,59.31%,85.47%,67.14%,-15.95%,+28.39%,68.54%,neutral
8,Gift-buyer general Eid,checkout_readiness,23,34,same_objective_empirical,15,13.942,6.622,36.942,17.622,67.65%,67.70%,55.55%,79.41%,67.80%,-21.99%,+24.22%,48.68%,neutral
9,1% Lookalike past gift buyers Eid,checkout_readiness,7,11,same_objective_empirical,15,14.178,6.702,21.178,10.702,63.64%,66.43%,49.93%,81.52%,67.90%,-25.78%,+24.59%,44.00%,neutral


**Ad: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Cairo broad creative A initial,checkout_readiness,11,27,same_objective_empirical,27,9.659,4.229,20.659,20.229,40.74%,50.53%,35.35%,65.12%,69.55%,-46.03%,+11.34%,9.66%,neutral
1,Cairo broad creative B spring refresh,checkout_readiness,1,2,same_objective_empirical,27,9.580,4.570,10.580,5.570,50.00%,65.51%,41.35%,85.63%,67.71%,-35.37%,+31.13%,44.86%,insufficient_evidence
2,Cairo broad creative C summer refresh,checkout_readiness,6,12,same_objective_empirical,27,9.567,4.470,15.567,10.470,50.00%,59.79%,40.96%,77.64%,68.16%,-37.39%,+22.33%,27.16%,neutral
3,1% LAL acquisition initial,checkout_readiness,15,27,same_objective_empirical,27,9.124,4.199,24.124,16.199,55.56%,59.83%,44.35%,74.18%,68.48%,-35.67%,+21.99%,26.86%,neutral
4,1% LAL acquisition refresh,checkout_readiness,11,16,same_objective_empirical,27,8.821,4.233,19.821,9.233,68.75%,68.22%,50.45%,83.55%,67.57%,-27.47%,+30.65%,50.84%,neutral
5,3% LAL scale,checkout_readiness,12,15,same_objective_empirical,27,9.243,4.524,21.243,7.524,80.00%,73.85%,56.42%,88.13%,67.14%,-20.49%,+37.71%,66.16%,neutral
6,Ramadan Cairo women creative A,checkout_readiness,14,16,same_objective_empirical,27,9.549,4.747,23.549,6.747,87.50%,77.73%,61.76%,90.55%,66.80%,-15.72%,+39.91%,78.04%,neutral
7,Ramadan Cairo women creative B refresh,checkout_readiness,17,23,same_objective_empirical,27,8.934,4.353,25.934,10.353,73.91%,71.47%,56.17%,84.76%,67.24%,-22.28%,+33.67%,59.98%,neutral
8,Ramadan multi-city creative A,checkout_readiness,7,10,same_objective_empirical,27,8.835,4.243,15.835,7.243,70.00%,68.62%,48.79%,85.27%,67.56%,-28.54%,+32.12%,51.94%,neutral
9,Ramadan multi-city creative B refresh,checkout_readiness,8,12,same_objective_empirical,27,8.813,4.215,16.813,8.215,66.67%,67.18%,47.76%,83.65%,67.65%,-30.23%,+32.03%,47.28%,neutral


**Creative: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Always-on premium acquisition default,checkout_readiness,22,43,same_objective_empirical,20,10.042,4.389,32.042,25.389,51.16%,55.79%,42.97%,68.54%,69.58%,-37.98%,+14.70%,15.54%,neutral
1,Always-on premium spring refresh,checkout_readiness,1,2,same_objective_empirical,20,10.281,4.904,11.281,5.904,50.00%,65.65%,43.03%,85.29%,67.71%,-33.62%,+29.72%,44.48%,insufficient_evidence
2,Always-on premium summer refresh,checkout_readiness,6,12,same_objective_empirical,20,10.268,4.797,16.268,10.797,50.00%,60.11%,41.82%,77.72%,68.16%,-35.55%,+23.10%,29.56%,neutral
3,Lookalike scale premium quality,checkout_readiness,15,27,same_objective_empirical,20,9.609,4.422,24.609,16.422,55.56%,59.98%,44.62%,74.02%,68.48%,-34.10%,+20.93%,26.16%,neutral
4,Lookalike scale value subscription,checkout_readiness,12,15,same_objective_empirical,20,9.875,4.833,21.875,7.833,80.00%,73.63%,57.14%,87.48%,67.14%,-19.70%,+35.38%,66.70%,neutral
5,Suhoor essentials Ramadan,checkout_readiness,21,26,same_objective_empirical,20,9.969,4.975,30.969,9.975,80.77%,75.64%,61.25%,87.42%,66.71%,-16.55%,+36.07%,72.32%,neutral
6,Ramadan tradition modernized health,checkout_readiness,17,23,same_objective_empirical,20,9.375,4.568,26.375,10.568,73.91%,71.39%,56.08%,84.47%,67.24%,-21.88%,+32.88%,58.92%,neutral
7,Ramadan Cocoa Spice limited edition,checkout_readiness,8,12,same_objective_empirical,20,9.170,4.386,17.170,8.386,66.67%,67.19%,47.82%,83.99%,67.65%,-29.39%,+30.57%,47.72%,neutral
8,Iftar premium bundle,checkout_readiness,23,38,same_objective_empirical,20,9.286,4.299,32.286,19.299,60.53%,62.59%,49.43%,75.59%,68.36%,-30.63%,+23.30%,33.30%,neutral
9,Iftar bundle scarcity,checkout_readiness,23,29,same_objective_empirical,20,9.806,4.893,32.806,10.893,79.31%,75.07%,61.95%,86.33%,66.71%,-16.99%,+37.04%,71.72%,neutral


**Audience: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Broad,checkout_readiness,18,41,same_objective_empirical,13,19.523,8.170,37.523,31.170,43.90%,54.62%,42.86%,65.82%,70.50%,-34.97%,+4.41%,6.64%,neutral
1,Lookalike,checkout_readiness,38,57,same_objective_empirical,13,14.440,6.804,52.440,25.804,66.67%,67.02%,56.25%,76.92%,67.97%,-21.53%,+22.03%,43.64%,neutral
2,Demographic,checkout_readiness,31,39,same_objective_empirical,13,16.764,8.434,47.764,16.434,79.49%,74.40%,63.52%,84.15%,66.53%,-12.44%,+30.00%,77.44%,neutral
3,Broad,checkout_readiness,15,22,same_objective_empirical,13,14.445,6.872,29.445,13.872,68.18%,67.98%,53.93%,81.07%,67.76%,-22.14%,+24.97%,50.42%,neutral
4,Demographic,checkout_readiness,23,35,same_objective_empirical,13,14.474,6.816,37.474,18.816,65.71%,66.57%,54.15%,78.31%,67.98%,-22.68%,+22.84%,45.00%,neutral
5,Lookalike,checkout_readiness,15,23,same_objective_empirical,13,14.503,6.843,29.503,14.843,65.22%,66.53%,52.05%,79.26%,67.94%,-24.49%,+22.55%,44.66%,neutral
6,Retargeting,checkout_readiness,14,17,same_objective_empirical,13,18.298,8.954,32.298,11.954,82.35%,72.99%,59.18%,84.79%,67.14%,-15.86%,+27.31%,68.70%,neutral
7,Broad,checkout_readiness,23,34,same_objective_empirical,13,14.437,6.857,37.437,17.857,67.65%,67.71%,54.88%,79.38%,67.80%,-22.33%,+23.84%,48.66%,neutral
8,Lookalike,checkout_readiness,7,11,same_objective_empirical,13,14.641,6.921,21.641,10.921,63.64%,66.46%,49.92%,81.72%,67.90%,-25.44%,+24.45%,45.24%,neutral
9,Demographic,checkout_readiness,22,29,same_objective_empirical,13,15.494,7.577,37.494,14.577,75.86%,72.01%,59.00%,83.22%,67.16%,-16.29%,+28.00%,65.46%,neutral


### 12.4 Compare Quality Ranges

Read the point together with the line. A high point with a wide range has limited
evidence. These ranges assume the extracted labels are correct; they do not account
for systematic LLM mistakes. Overlapping ranges should not be presented as proven
winners. Primary outcome scores remain separate because their denominators and
business meanings differ.

In [17]:
plotted = quality.dropna(subset=["corrected_rate"])
if not plotted.empty:
    base = alt.Chart(plotted).encode(y=alt.Y("entity_name:N", title=None))
    bounds = base.mark_rule(strokeWidth=4, color="#218380").encode(
        x=alt.X("range_low:Q", title="Conversation quality", axis=alt.Axis(format=".0%"), scale=alt.Scale(domain=[0, 1])),
        x2="range_high:Q",
        tooltip=["entity_name", "successes", "trials", "prior_source", "status"]
    )
    points = base.mark_point(filled=True, color="#20262e", size=90).encode(x="corrected_rate:Q")
    display((bounds + points).properties(width=760, height=max(150, 45 * len(plotted))))

alt.LayerChart(...)

### 12.5 Check Whether Quality Is Associated With Outcomes

The next table compares orders and deliveries between semantic-positive,
semantic-negative and unknown selected conversations. It is a retrospective
association, not causal evidence or an independent predictive evaluation: the
extractor saw the full conversation, including historical checkout steps.
This is a starting point for manually reviewing whether the definitions are useful.
Small groups and missing labels can make the differences unstable.

In [18]:
selected = campaign_audit[campaign_audit["selected"]].copy()
selected["Semantic result"] = selected["success"].map({True: "positive", False: "negative"}).fillna("unknown")
associations = selected.groupby(["campaign_id", "Semantic result"]).agg(
    customers=("conversation_id", "size"), orders=("has_order", "sum"), deliveries=("is_delivered", "sum")
).reset_index()
associations["order_rate"] = associations["orders"] / associations["customers"]
associations["delivery_rate"] = associations["deliveries"] / associations["customers"]
display(associations.style.format({"order_rate": "{:.2%}", "delivery_rate": "{:.2%}"}))

,campaign_id,Semantic result,customers,orders,deliveries,order_rate,delivery_rate
0,120209876543220001,negative,42,12,6,28.57%,14.29%
1,120209876543220001,positive,56,55,44,98.21%,78.57%
2,120209876543220001,unknown,5,2,2,40.00%,40.00%
3,120209876543220005,negative,15,2,2,13.33%,13.33%
4,120209876543220005,positive,46,44,39,95.65%,84.78%
5,120209876543220005,unknown,1,0,0,0.00%,0.00%
6,120209876543220006,negative,23,13,3,56.52%,13.04%
7,120209876543220006,positive,52,51,39,98.08%,75.00%
8,120209876543220006,unknown,1,0,0,0.00%,0.00%
9,120209876543220007,negative,15,10,5,66.67%,33.33%


### 12.6 How Primary And Conversation Evidence Allocate Budget

Every campaign with both required components receives a share of its objective's
full envelope:

```text
priority = primary probability_better x semantic 95% lower bound
weight = priority / sum of campaign priorities in the objective
campaign budget = full objective envelope x weight
```

Objective envelopes retain previous-cycle spend shares because their primary and
semantic metrics are not comparable across objectives. Within an objective, action
labels do not gate this POC allocation. Missing either component gives the campaign
zero allocation. The priority product is a policy index, not a joint probability or
a learned return-on-spend optimum.

Review labels against transcripts before operational use. No manual validation is
claimed here. The table shows exactly how much this assumption changes the budget.

In [19]:
impact = pd.DataFrame([
    {"Campaign": item.campaign_name,
     "Primary probability": item.primary_probability_component,
     "Quality lower bound": item.semantic_priority,
     "Priority product": item.allocation_priority,
     "Normalized weight": item.allocation_weight,
     "Basis": item.allocation_basis,
     "Previous-spend scenario": item.previous_spend_budget_units,
     "Two-factor scenario": item.recommended_budget_units,
     "Change": item.recommended_budget_units - item.previous_spend_budget_units}
    for item in result.allocations if item.objective == FOCUS_OBJECTIVE
])
display(impact.style.format({
    "Primary probability": "{:.2%}", "Quality lower bound": "{:.2%}",
    "Priority product": "{:.4f}", "Normalized weight": "{:.2%}",
    "Previous-spend scenario": "{:.3f}", "Two-factor scenario": "{:.3f}",
    "Change": "{:+.3f}"
}, na_rep="Not available"))

,Campaign,Primary probability,Quality lower bound,Priority product,Normalized weight,Basis,Previous-spend scenario,Two-factor scenario,Change
0,Always-On Premium Acquisition,13.24%,54.89%,0.0727,4.01%,primary_probability_x_semantic_lower_bound,34.997,3.304,-31.693
1,Ramadan Suhoor Specials,68.92%,62.63%,0.4317,23.83%,primary_probability_x_semantic_lower_bound,8.028,19.626,+11.598
2,Ramadan Iftar Premium Bundles,32.78%,59.95%,0.1965,10.85%,primary_probability_x_semantic_lower_bound,14.193,8.935,-5.258
3,Eid Gifting Premium,51.10%,56.63%,0.2894,15.98%,primary_probability_x_semantic_lower_bound,3.723,13.158,+9.435
4,Summer Premium Launch,87.22%,64.19%,0.5599,30.91%,primary_probability_x_semantic_lower_bound,17.052,25.455,+8.403
5,Mid-Year Sale,47.74%,54.72%,0.2612,14.42%,primary_probability_x_semantic_lower_bound,4.363,11.878,+7.515


## 13. Campaign-By-Campaign Recommendation

The statements below translate the locked deterministic outputs. They do not create a
new decision from the semantic signals.

In [20]:
sections = []
for _, row in campaigns.sort_values("spend", ascending=False).iterrows():
    allocation = allocations[str(row["campaign_id"])]
    probability = (
        f"{row['probability_better']:.1%}"
        if pd.notna(row["probability_better"])
        else "not available"
    )
    lift = (
        f"[{row['lift_low']:+.1%}, {row['lift_high']:+.1%}]"
        if pd.notna(row["lift_low"]) else "not available"
    )
    portfolio_context = (
        f"{row['portfolio_context_benchmark']:.1%}"
        if pd.notna(row["portfolio_context_benchmark"])
        else "not available"
    )
    semantic_sentence = (
        f"Semantic coverage is {row['semantic_coverage_rate']:.1%}; the leading "
        f"barrier is {row['top_barrier'] or 'not established'} and the leading "
        f"value driver is {row['top_value_driver'] or 'not established'}."
        if pd.notna(row["semantic_coverage_rate"])
        else "Conversation semantics are not yet available for this campaign."
    )
    sections.append(
        f"### {row['campaign_name']}\n"
        f"- **Action:** `{row['recommended_action'].upper()}`; "
        f"statistical decision `{row['statistical_decision'].upper()}`.\n"
        f"- **Evidence:** {row['score_successes']:.0f} successes from "
        f"{row['score_trials']:.0f} eligible trials; raw {row['raw_rate']:.1%}.\n"
        f"- **Uncertainty:** probability better {probability}; lift range {lift}.\n"
        f"- **Benchmark:** \`{row['benchmark_scope']}\` with "
        f"\`{row['benchmark_quality']}\` quality; portfolio context "
        f"{portfolio_context} is descriptive only.\n"
        f"- **Efficiency:** {row['efficiency_metric'].replace('_', ' ')} is "
        f"`{row['efficiency_comparison']}`.\n"
        f"- **Budget:** {allocation.recommended_budget_units:.2f} "
        f"{result.recommendation_input.cycle.currency} units; priority is "
        f"{allocation.primary_probability_component:.1%} x "
        f"{allocation.semantic_priority:.1%}.\n"
        f"- **Conversation context:** {semantic_sentence}\n"
        f"- **Reason codes:** {', '.join(allocation.reason_codes)}."
    )
display(Markdown("\n\n".join(sections)))

### Always-On Premium Acquisition
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 52 successes from 103 eligible trials; raw 50.5%.
- **Uncertainty:** probability better 13.2%; lift range [-22.6%, +6.4%].
- **Benchmark:** \`same_objective\` with \`decision_grade\` quality; portfolio context 59.2% is descriptive only.
- **Efficiency:** net roas is `worse_than_peer`.
- **Budget:** 3.30 units units; priority is 13.2% x 54.9%.
- **Conversation context:** Semantic coverage is 100.0%; the leading barrier is price and the leading value driver is price.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

### Summer Premium Launch
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 64 successes from 90 eligible trials; raw 71.1%.
- **Uncertainty:** probability better 87.2%; lift range [-6.0%, +22.5%].
- **Benchmark:** \`same_objective\` with \`decision_grade\` quality; portfolio context 55.2% is descriptive only.
- **Efficiency:** net roas is `worse_than_peer`.
- **Budget:** 25.46 units units; priority is 87.2% x 64.2%.
- **Conversation context:** Semantic coverage is 100.0%; the leading barrier is price and the leading value driver is gifting.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

### Ramadan Iftar Premium Bundles
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 42 successes from 76 eligible trials; raw 55.3%.
- **Uncertainty:** probability better 32.8%; lift range [-21.3%, +13.9%].
- **Benchmark:** \`same_objective\` with \`decision_grade\` quality; portfolio context 58.0% is descriptive only.
- **Efficiency:** net roas is `better_than_peer`.
- **Budget:** 8.94 units units; priority is 32.8% x 60.0%.
- **Conversation context:** Semantic coverage is 100.0%; the leading barrier is timing and the leading value driver is price.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

### Ramadan Suhoor Specials
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 41 successes from 62 eligible trials; raw 66.1%.
- **Uncertainty:** probability better 68.9%; lift range [-13.4%, +22.1%].
- **Benchmark:** \`same_objective\` with \`decision_grade\` quality; portfolio context 56.6% is descriptive only.
- **Efficiency:** net roas is `worse_than_peer`.
- **Budget:** 19.63 units units; priority is 68.9% x 62.6%.
- **Conversation context:** Semantic coverage is 98.4%; the leading barrier is delivery and the leading value driver is delivery_speed.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

### Mid-Year Sale
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 20 successes from 34 eligible trials; raw 58.8%.
- **Uncertainty:** probability better 47.7%; lift range [-20.1%, +19.5%].
- **Benchmark:** \`same_objective\` with \`decision_grade\` quality; portfolio context 57.6% is descriptive only.
- **Efficiency:** net roas is `better_than_peer`.
- **Budget:** 11.88 units units; priority is 47.7% x 54.7%.
- **Conversation context:** Semantic coverage is 97.1%; the leading barrier is product_fit and the leading value driver is price.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

### Eid Gifting Premium
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 28 successes from 46 eligible trials; raw 60.9%.
- **Uncertainty:** probability better 51.1%; lift range [-18.7%, +20.4%].
- **Benchmark:** \`same_objective\` with \`decision_grade\` quality; portfolio context 57.4% is descriptive only.
- **Efficiency:** net roas is `better_than_peer`.
- **Budget:** 13.16 units units; priority is 51.1% x 56.6%.
- **Conversation context:** Semantic coverage is 100.0%; the leading barrier is product_quality and the leading value driver is gifting.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

## 14. Objective Summary For Stakeholders

This final view keeps three audiences aligned:

- **Business owner:** allocated and unallocated budget, outcome quality, and economics.
- **Marketing director:** objective achievement, portfolio evidence, and strategic tests.
- **Performance marketing manager:** campaign, adset, ad, creative, and audience actions.

In [21]:
action_counts = campaigns["recommended_action"].value_counts().to_dict()
objective_budget = sum(
    allocations[str(campaign_id)].recommended_budget_units
    for campaign_id in campaigns["campaign_id"]
)
objective_spend_share = campaigns["spend"].sum() / result.scorecards["campaign"]["spend"].sum()
objective_envelope = policy.budget_units * objective_spend_share
objective_unallocated = objective_envelope - objective_budget
semantic_coverage = (
    campaigns["semantic_conversations"].sum()
    / campaigns["observed_conversations"].sum()
    if campaigns["observed_conversations"].sum() else np.nan
)

summary = pd.DataFrame([
    ["Campaigns", len(campaigns)],
    ["Final actions", action_counts],
    ["Previous-cycle objective spend share", objective_spend_share],
    ["Objective budget envelope", objective_envelope],
    ["Allocated to this objective", objective_budget],
    ["Unallocated in this objective", objective_unallocated],
    ["Conversation semantic coverage", semantic_coverage],
    ["Funded named tests", len(objective_tests)],
], columns=["Summary item", "Value"])
display(summary.style.hide(axis="index"))

display(Markdown(
    "**Decision discipline:** keep final actions as range-based reporting labels, "
    "allocate this POC's full objective envelopes with the explicit two-factor "
    "priority, and interpret overlapping ranges as uncertain rather than proven "
    "superiority."
))

Summary item,Value
Campaigns,6
Final actions,{'keep_as_test': 6}
Previous-cycle objective spend share,0.823560
Objective budget envelope,82.355955
Allocated to this objective,82.355955
Unallocated in this objective,0.000000
Conversation semantic coverage,0.995423
Funded named tests,6


**Decision discipline:** keep final actions as range-based reporting labels, allocate this POC's full objective envelopes with the explicit two-factor priority, and interpret overlapping ranges as uncertain rather than proven superiority.